In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdchem import Mol

In [3]:
def Test(smiles:str):#test_generation_conformers
    mol: Mol = Chem.MolFromSmiles(smiles, sanitize=True)
    if mol == None:
        raise ValueError("Invalid SMILES")
    else:
        mol = Chem.AddHs(mol)
        Confs_ids=AllChem.EmbedMultipleConfs(mol, numConfs=100)
        n=mol.GetNumConformers()
        
    return n
print(Test("C-C"))

100


In [4]:
def mol_from_SMILES(smiles:str):#treatment of SMILES input
    mol: Mol = Chem.MolFromSmiles(smiles, sanitize=True)
    if mol == None:
        raise ValueError("Invalid SMILES")
    else:
        mol = Chem.AddHs(mol)
        return mol

In [5]:
def conformer_selection(mol:Mol, num_confs:int, filename_1:str):#Generation of conformers and selection of the most stable one
    conf_ids:list[int] = AllChem.EmbedMultipleConfs(mol, numConfs=num_confs)
    if not conf_ids:
        raise ValueError("No conformers generated")
    energies:dict[int, float] = {}
    for conf_id in conf_ids:
        ff = AllChem.UFFGetMoleculeForceField(mol, confId=conf_id)
        ff.Minimize()
        energies[conf_id] = ff.CalcEnergy()
    most_stable_conformer:int = min(energies, key=energies.get)
    mol_block:str=Chem.MolToMolBlock(mol,confId=most_stable_conformer)
    with open(filename_1, "w") as file:
        file.write(mol_block)
    return most_stable_conformer

In [6]:
def overall_conversion(Smiles:str,filename_1:str, filename_2:str, num_confs:int):#Obtention of the xyz file of the most stable conformer
    mol:Mol=mol_from_SMILES(Smiles)
    most_stable_conformer:int=conformer_selection(mol, num_confs, filename_1)
    xyz:str = Chem.MolToXYZBlock(mol, confId=most_stable_conformer)
    with open(filename_2, "w") as file:
        file.write(xyz)
    return xyz

In [7]:
smiles="C-C-C-C"
mol=mol_from_SMILES(smiles)
conf_ids:list[int] = AllChem.EmbedMultipleConfs(mol, numConfs=10)
ff=AllChem.UFFGetMoleculeForceField(mol, confId=7)
print(ff)

In [14]:
print(overall_conversion("C-C-O", "Ethanol.SDF", "Ethanol.xyz",10000))

9

C      0.979053    0.307919   -0.060281
C     -0.374640   -0.368902    0.089201
O     -1.393288    0.554217   -0.175621
H      1.062578    1.153346    0.655234
H      1.788585   -0.422689    0.149385
H      1.099817    0.692230   -1.095368
H     -0.476260   -0.757437    1.126964
H     -0.438975   -1.219122   -0.625799
H     -2.246869    0.060438   -0.063715



In [10]:
help(Chem.MolToMolFile)

Help on built-in function MolToMolFile in module rdkit.Chem.rdmolfiles:

MolToMolFile(...)
    MolToMolFile( (Mol)mol, (str)filename, (MolWriterParams)params [, (int)confId=-1]) -> None :
        Writes a Mol file for a molecule
          ARGUMENTS:
        
            - mol: the molecule
            - filename: the file to write to
            - params: the MolWriterParams
            - confId: (optional) selects which conformation to output (-1 = default)
        
        
    
        C++ signature :
            void MolToMolFile(class RDKit::ROMol,class std::basic_string<char,struct std::char_traits<char>,class std::allocator<char> >,struct RDKit::MolWriterParams [,int=-1])
    
    MolToMolFile( (Mol)mol, (str)filename [, (bool)includeStereo=True [, (int)confId=-1 [, (bool)kekulize=True [, (bool)forceV3000=False]]]]) -> None :
        Writes a Mol file for a molecule
          ARGUMENTS:
        
            - mol: the molecule
            - filename: the file to write to
       

In [11]:
help(Chem.MolToXYZFile)

Help on built-in function MolToXYZFile in module rdkit.Chem.rdmolfiles:

MolToXYZFile(...)
    MolToXYZFile( (Mol)mol, (str)filename [, (int)confId=-1 [, (int)precision=6]]) -> None :
        Writes a XYZ file for a molecule
          ARGUMENTS:
        
            - mol: the molecule
            - filename: the file to write to
            - confId: (optional) selects which conformation to output (-1 = default)
            - precision: precision of the coordinates
        
        
    
        C++ signature :
            void MolToXYZFile(class RDKit::ROMol,class std::basic_string<char,struct std::char_traits<char>,class std::allocator<char> > [,int=-1 [,unsigned int=6]])

